In [ ]:
from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS, UNKNOWN_LABEL
from omegaconf import OmegaConf
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, balanced_accuracy_score, roc_auc_score

dataset_path = "/opt/gpudata/ecg/heedb"
arms = ["2D-partial", "2D-global", "1D-global"]
run_path = Path("/opt/gpu_working/steven/protoecgnet-heedb-v4")
# ground truth labels: "original_physician" or "audited_physician" (has unknowns, which are
# left out per label). MUSE predictions are read the same way as the ground truth:
# audited_muse only counts main-read codes, original_muse also counts the comparison text
label_src = "audited_physician"
muse_label_src = "original_muse" if label_src == "original_physician" else "audited_muse"
config_path = Path("/opt/gpu_working/steven/ProtoSSL/large-user-study/configs")
fusion_path = run_path / "fusion/train-fusion-classifier/latest"
fusion_labels = list(HEEDB_TARGETS)
label_idxs = np.arange(len(HEEDB_TARGETS))

In [ ]:
def stack_branched_arms():
    arm_labels = [l for arm in arms for l in OmegaConf.load(config_path / f"{arm}.yaml").data.init_args.label_subset]
    idxs = np.asarray([arm_labels.index(l) for l in fusion_labels]) # indices to normalize to fusion order

    ret = dict()
    for split in ["val", "test"] :
        arm_probs = {arm: np.load(run_path / arm / f"train-classifier/latest/{split}_probs.npy") for arm in arms}
        stacked_probs = np.concat([arm_probs[arm] for arm in arms], axis=1)
        ret[split] = stacked_probs[:, idxs]
    return ret

In [ ]:
ds_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src=label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

ds_phys_val = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    label_src=label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

ds_muse = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src=muse_label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

In [ ]:
y_true = ds_phys.labels.numpy()[:, label_idxs]
y_true_val = ds_phys_val.labels.numpy()[:, label_idxs]
y_pred_muse = ds_muse.labels.numpy()[:, label_idxs]
# per-label masks of ECGs with a known ground truth label
known = y_true != UNKNOWN_LABEL
known_val = y_true_val != UNKNOWN_LABEL
assert (y_pred_muse != UNKNOWN_LABEL).all()

In [ ]:
def compute_threshold(_y_true, _y_prob):
    # NOTE: should be computed on either train/val set, NOT test
    fpr, tpr, thresholds = roc_curve(_y_true, _y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold

In [ ]:
temp = stack_branched_arms()
y_prob_branched, y_prob_branched_val = temp["test"], temp["val"]
y_prob_fusion = np.load(fusion_path / "test_probs.npy")
y_prob_fusion_val = np.load(fusion_path / "val_probs.npy")
assert y_prob_fusion.shape[1] == len(fusion_labels)

y_pred_branched = np.zeros_like(y_pred_muse)
y_pred_fusion = np.zeros_like(y_pred_muse)
accuracy_results = []
roc_results = []
for i, label in enumerate(fusion_labels):
    kv, k = known_val[:, i], known[:, i]
    branched_threshold = compute_threshold(y_true_val[kv, i], y_prob_branched_val[kv, i])
    y_pred_branched[:, i] = (y_prob_branched[:, i] > branched_threshold).astype(int)
    fusion_threshold = compute_threshold(y_true_val[kv, i], y_prob_fusion_val[kv, i])
    y_pred_fusion[:, i] = (y_prob_fusion[:, i] > fusion_threshold).astype(int)
    label_acc_results = {
        "label": label,
        "muse: BalAcc": balanced_accuracy_score(y_true[k, i], y_pred_muse[k, i]),
        "branched: BalAcc": balanced_accuracy_score(y_true[k, i], y_pred_branched[k, i]),
        "fusion: BalAcc": balanced_accuracy_score(y_true[k, i], y_pred_fusion[k, i]),
    }
    label_roc_results = {
        "label": label,
        # NOTE: can't compute muse roc given no probs
        "branched: AUROC": roc_auc_score(y_true[k, i], y_prob_branched[k, i]),
        "fusion: AUROC": roc_auc_score(y_true[k, i], y_prob_fusion[k, i]),
    }
    accuracy_results.append(label_acc_results)
    roc_results.append(label_roc_results)

In [ ]:
roc_df = pd.DataFrame(roc_results)
idx = len(roc_df)
roc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in roc_df.columns if c != "label"]:
    roc_df.loc[idx, c] = roc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(roc_df)

In [ ]:
acc_df = pd.DataFrame(accuracy_results)
idx = len(acc_df)
acc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in acc_df.columns if c != "label"]:
    acc_df.loc[idx, c] = acc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(acc_df)